# Hybrid BCI Controller: Combining Jaw Clicks and EEG Directional Intent

## Section 1 — Introduction

The final project is a **hybrid BCI control system** rather than a single-model EEG classifier. Jaw activity serves as the strongest discrete click signal, while EEG left/right decoding is treated as a weaker directional intent branch.

In the live runtime, the system combines:

- fixed saved model artifacts
- session-specific baseline calibration
- threshold and margin adaptation
- smoothing and cooldown logic
- GUI/game control output

The goal is not only offline accuracy. The real target is **usable real-time interaction**.


In [ ]:
import io
import json
import os
import pickle
import re
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from google.colab import files

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "figure.dpi": 120,
        "axes.facecolor": "#fbfbfd",
        "figure.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
    }
)

DEFAULT_FS_FALLBACK = 250.0
EXPECTED_BUNDLE_FILES = {
    "run_metadata.json": "Run-level metadata for the live or replay session.",
    "classifier_provenance.json": "Saved-model provenance and feature information for the runtime.",
    "adaptation_summary.json": "Baseline, thresholds, margins, and session-specific adaptation values.",
    "phase_log.csv": "Phase-level timing for baseline, cue, guided collection, and gameplay.",
    "protocol_truth.csv": "Prompted actions or expected protocol intervals.",
    "marker_log.csv": "Recorded marker events from the acquisition/runtime layer.",
    "control_trace.csv": "Continuous control confidences or command scores over time.",
    "control_events.csv": "Discrete control events emitted by the runtime.",
    "game_trace.csv": "Continuous game-state trace over time.",
    "game_events.csv": "Discrete game events such as actions, hits, or misses.",
    "session_feedback.json": "Human-readable feedback summary for the session.",
}
RESULT_HINTS = ("summary", "result", "benchmark", "report", "coverage", "metric")
RAW_RECORDING_HINTS = ("LR", "LRJ", "HR", "JAW", "LEFT", "RIGHT", "OPENBCI")
TIME_HINTS = ("time", "timestamp", "sec", "seconds", "ms", "millis")


## Section 2 — Upload Files

Upload any combination of raw recordings, live run logs, model artifacts, metadata JSON files, and result summaries. The notebook groups them into professor-readable categories rather than assuming a perfect run bundle.


In [ ]:
print("Upload raw recordings, live run files, models, and summaries.")
uploaded_raw = files.upload()

if not uploaded_raw:
    raise RuntimeError("No files were uploaded. Re-run this cell and choose one or more files.")

UPLOADED_FILES = {name: blob for name, blob in uploaded_raw.items()}
UPLOADED_FILENAMES = sorted(UPLOADED_FILES.keys())


def base_name(name):
    return os.path.basename(name)


def lower_name(name):
    return base_name(name).lower()


def classify_uploaded_file(name):
    lower = lower_name(name)
    if lower in EXPECTED_BUNDLE_FILES:
        return "live run log" if lower.endswith('.csv') else "metadata json"
    if lower.endswith((".pkl", ".pickle", ".joblib")):
        return "model artifact"
    if lower.endswith(".json"):
        return "metadata json"
    if lower.endswith(".md") or any(hint in lower for hint in RESULT_HINTS):
        return "result summary"
    if lower.endswith((".csv", ".tsv", ".txt")):
        if any(token.lower() in lower for token in ["control_trace", "control_events", "game_trace", "game_events", "phase_log", "protocol_truth", "marker_log"]):
            return "live run log"
        if any(token.lower() in lower for token in RAW_RECORDING_HINTS):
            return "raw recording"
        return "csv/other"
    return "other"


FILE_GROUPS = defaultdict(list)
for filename in UPLOADED_FILENAMES:
    FILE_GROUPS[classify_uploaded_file(filename)].append(filename)

upload_table = pd.DataFrame(
    {
        "Index": np.arange(1, len(UPLOADED_FILENAMES) + 1),
        "Filename": UPLOADED_FILENAMES,
        "Group": [classify_uploaded_file(name) for name in UPLOADED_FILENAMES],
        "Size (KB)": [round(len(UPLOADED_FILES[name]) / 1024.0, 1) for name in UPLOADED_FILENAMES],
    }
).sort_values(["Group", "Filename"]).reset_index(drop=True)
display(upload_table)

print("Grouped upload summary:")
for group_name in sorted(FILE_GROUPS):
    print(f"- {group_name}: {len(FILE_GROUPS[group_name])} file(s)")


## Section 3 — Runtime Architecture Overview

The final runtime combines fixed saved models with session-specific stabilization logic.

```text
OpenBCI stream
  -> preprocessing
  -> jaw click model
  -> EEG left/right model
  -> baseline calibration
  -> threshold/margin adaptation
  -> control event logic
  -> GUI/game
```

Control mapping:

- **jaw = click / select**
- **EEG left = move left**
- **EEG right = move right**
- **smoothing and cooldown = fewer noisy repeated actions**

This architecture matters because the project is not just “run a classifier.” It is a control system that must behave sensibly in real time.


## Section 4 — Parse Live Run Bundle

If live-run bundle files are present, the notebook loads whichever ones are available. It does **not** assume every file exists. For each detected file, the notebook shows a small summary and briefly explains its role in the runtime.


In [ ]:
def read_uploaded_csv(file_bytes):
    text = file_bytes.decode("utf-8", errors="ignore")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return pd.read_csv(io.StringIO(text))


def read_uploaded_json(file_bytes):
    text = file_bytes.decode("utf-8", errors="ignore")
    return json.loads(text)


def find_uploaded_file(target_basename):
    target = target_basename.lower()
    for filename in UPLOADED_FILENAMES:
        if lower_name(filename) == target:
            return filename
    return None


BUNDLE_OBJECTS = {}
BUNDLE_SUMMARY_ROWS = []

for expected_name, role_description in EXPECTED_BUNDLE_FILES.items():
    matched_file = find_uploaded_file(expected_name)
    if matched_file is None:
        continue

    try:
        if expected_name.endswith('.csv'):
            obj = read_uploaded_csv(UPLOADED_FILES[matched_file])
            object_type = 'csv'
            key_summary = f"{obj.shape[0]} rows x {obj.shape[1]} cols"
        else:
            obj = read_uploaded_json(UPLOADED_FILES[matched_file])
            object_type = 'json'
            if isinstance(obj, dict):
                key_summary = f"{len(obj)} top-level keys"
            elif isinstance(obj, list):
                key_summary = f"list with {len(obj)} items"
            else:
                key_summary = type(obj).__name__
        BUNDLE_OBJECTS[expected_name] = obj
        BUNDLE_SUMMARY_ROWS.append(
            {
                'Bundle File': expected_name,
                'Uploaded Name': matched_file,
                'Type': object_type,
                'Summary': key_summary,
                'Role': role_description,
            }
        )
    except Exception as exc:
        warnings.warn(f"Could not parse {matched_file}: {exc}")

if BUNDLE_SUMMARY_ROWS:
    BUNDLE_SUMMARY_DF = pd.DataFrame(BUNDLE_SUMMARY_ROWS)
    display(BUNDLE_SUMMARY_DF)

    for row in BUNDLE_SUMMARY_ROWS:
        print(f"\n{row['Bundle File']} — {row['Role']}")
        obj = BUNDLE_OBJECTS[row['Bundle File']]
        if isinstance(obj, pd.DataFrame):
            display(obj.head())
        elif isinstance(obj, dict):
            preview_df = pd.DataFrame({'Key': list(obj.keys())[:15], 'Value Preview': [str(obj[key])[:120] for key in list(obj.keys())[:15]]})
            display(preview_df)
        elif isinstance(obj, list):
            display(pd.DataFrame({'Item Preview': [str(item)[:160] for item in obj[:10]]}))
else:
    print('No recognized live-run bundle files were uploaded.')


## Section 5 — Metadata and Provenance Summary

The runtime JSON files provide the most direct explanation of what the live system was actually using: model provenance, adaptation parameters, and session metadata.


In [ ]:
def flatten_json(obj, prefix=""):
    items = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = f"{prefix}.{key}" if prefix else str(key)
            items.update(flatten_json(value, new_prefix))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            new_prefix = f"{prefix}[{idx}]"
            items.update(flatten_json(value, new_prefix))
    else:
        items[prefix] = obj
    return items


def extract_rows_from_flat_map(flat_map, keywords, fallback_title):
    rows = []
    lowered_keywords = [kw.lower() for kw in keywords]
    for key, value in flat_map.items():
        if any(keyword in key.lower() for keyword in lowered_keywords):
            rows.append({"Field": key, "Value": str(value)})
    if not rows:
        rows = [{"Field": key, "Value": str(value)} for key, value in list(flat_map.items())[:15]]
    return pd.DataFrame(rows)


if 'classifier_provenance.json' in BUNDLE_OBJECTS:
    print('classifier_provenance.json summary')
    flat_map = flatten_json(BUNDLE_OBJECTS['classifier_provenance.json'])
    display(extract_rows_from_flat_map(flat_map, ['model', 'train', 'source', 'feature', 'channel'], 'Classifier Provenance'))
else:
    print('classifier_provenance.json not available.')

if 'adaptation_summary.json' in BUNDLE_OBJECTS:
    print('adaptation_summary.json summary')
    flat_map = flatten_json(BUNDLE_OBJECTS['adaptation_summary.json'])
    display(extract_rows_from_flat_map(flat_map, ['baseline', 'threshold', 'margin', 'cooldown', 'rearm', 'adapt'], 'Adaptation Summary'))
else:
    print('adaptation_summary.json not available.')

if 'run_metadata.json' in BUNDLE_OBJECTS:
    print('run_metadata.json summary')
    flat_map = flatten_json(BUNDLE_OBJECTS['run_metadata.json'])
    display(extract_rows_from_flat_map(flat_map, ['date', 'time', 'mode', 'duration', 'participant', 'session', 'name'], 'Run Metadata'))
else:
    print('run_metadata.json not available.')


## Section 6 — Control Trace Visualization

If `control_trace.csv` is available, the plot below shows how continuous model outputs become runtime behavior. This is important because the live system does not act on raw probabilities directly; it uses thresholds, margins, and smoothing logic to decide when a control should actually fire.


In [ ]:
def find_time_column(df):
    columns = list(df.columns)
    normalized = {column: re.sub(r'[^a-z0-9]+', '_', str(column).strip().lower()) for column in columns}
    explicit = [column for column in columns if any(hint in normalized[column] for hint in TIME_HINTS)]
    if explicit:
        return explicit[0]
    numeric_columns = [column for column in columns if pd.to_numeric(df[column], errors='coerce').notna().mean() >= 0.95]
    if numeric_columns:
        return numeric_columns[0]
    return None


def find_probability_column(df, include_terms, exclude_terms=()):
    for column in df.columns:
        lower = str(column).lower()
        if all(term in lower for term in include_terms) and not any(term in lower for term in exclude_terms):
            return column
    for column in df.columns:
        lower = str(column).lower()
        if any(term in lower for term in include_terms) and not any(term in lower for term in exclude_terms):
            return column
    return None


def extract_threshold_values(adaptation_obj):
    if not isinstance(adaptation_obj, dict):
        return {}
    flat_map = flatten_json(adaptation_obj)
    thresholds = {}
    for key, value in flat_map.items():
        lower_key = key.lower()
        if not isinstance(value, (int, float)):
            continue
        if 'threshold' in lower_key or 'margin' in lower_key:
            thresholds[key] = float(value)
    return thresholds


if 'control_trace.csv' in BUNDLE_OBJECTS:
    control_trace_df = BUNDLE_OBJECTS['control_trace.csv'].copy()
    time_col = find_time_column(control_trace_df)
    jaw_col = find_probability_column(control_trace_df, ['jaw'], exclude_terms=['event'])
    left_col = find_probability_column(control_trace_df, ['left'], exclude_terms=['event'])
    right_col = find_probability_column(control_trace_df, ['right'], exclude_terms=['event'])
    command_col = find_probability_column(control_trace_df, ['command']) or find_probability_column(control_trace_df, ['action']) or find_probability_column(control_trace_df, ['selected'])

    print('Detected control_trace columns:')
    display(pd.DataFrame({
        'Role': ['time', 'jaw', 'left', 'right', 'command/action'],
        'Column': [time_col, jaw_col, left_col, right_col, command_col],
    }))

    if time_col is not None:
        fig, ax = plt.subplots(figsize=(14, 6))
        plotted_any = False
        for label, column, color in [('Jaw confidence', jaw_col, '#111827'), ('Left confidence', left_col, '#2563eb'), ('Right confidence', right_col, '#dc2626')]:
            if column is not None:
                ax.plot(pd.to_numeric(control_trace_df[time_col], errors='coerce'), pd.to_numeric(control_trace_df[column], errors='coerce'), label=label, linewidth=1.4, color=color)
                plotted_any = True

        thresholds = extract_threshold_values(BUNDLE_OBJECTS.get('adaptation_summary.json'))
        for key, value in list(thresholds.items())[:6]:
            if 0.0 <= value <= 1.5:
                ax.axhline(value, linestyle='--', linewidth=1.0, alpha=0.5, label=key.split('.')[-1])

        if command_col is not None:
            command_series = control_trace_df[command_col].astype(str)
            change_points = command_series.ne(command_series.shift()).fillna(False)
            change_df = control_trace_df.loc[change_points, [time_col, command_col]].copy()
            change_df = change_df.head(40)
            for row in change_df.itertuples(index=False):
                ax.axvline(getattr(row, time_col), color='#9ca3af', alpha=0.15)

        ax.set_title('Control Trace Overview')
        ax.set_xlabel(time_col)
        ax.set_ylabel('Confidence / score')
        if plotted_any:
            ax.legend(loc='upper right')
        plt.show()

        explanation = """
### Control Trace Interpretation

Continuous model outputs do not become actions immediately. The runtime uses calibration, thresholds, margins, smoothing, and cooldown logic to convert noisy confidence signals into stable control behavior.
"""
        display(Markdown(explanation))
    else:
        print('A usable time column was not detected in control_trace.csv.')
else:
    print('control_trace.csv not available.')


## Section 7 — Control Events Summary

If `control_events.csv` exists, the notebook summarizes the discrete runtime events emitted after smoothing and threshold logic. This is the most direct view of what the controller actually did.


In [ ]:
def find_event_type_column(df):
    preferred = ['event_type', 'type', 'action', 'command', 'event']
    lower_map = {str(column).lower(): column for column in df.columns}
    for candidate in preferred:
        if candidate in lower_map:
            return lower_map[candidate]
    for column in df.columns:
        lower = str(column).lower()
        if any(token in lower for token in ['event', 'action', 'command', 'type']):
            return column
    return None


if 'control_events.csv' in BUNDLE_OBJECTS:
    control_events_df = BUNDLE_OBJECTS['control_events.csv'].copy()
    time_col = find_time_column(control_events_df)
    event_col = find_event_type_column(control_events_df)

    if event_col is not None:
        summary_df = control_events_df.groupby(event_col).agg(
            Count=(event_col, 'size'),
            **({"First Time": (time_col, 'min'), "Last Time": (time_col, 'max')} if time_col is not None else {})
        ).reset_index().rename(columns={event_col: 'Event Type'})
        display(summary_df)

        if time_col is not None:
            event_codes = {event: idx for idx, event in enumerate(summary_df['Event Type'], start=1)}
            fig, ax = plt.subplots(figsize=(14, 4))
            for event_name, group_df in control_events_df.groupby(event_col):
                y_value = event_codes[event_name]
                x_vals = pd.to_numeric(group_df[time_col], errors='coerce')
                ax.vlines(x_vals, y_value - 0.35, y_value + 0.35, label=str(event_name), alpha=0.75)
            ax.set_yticks(list(event_codes.values()))
            ax.set_yticklabels(list(event_codes.keys()))
            ax.set_title('Control Event Timeline')
            ax.set_xlabel(time_col)
            ax.set_ylabel('Event Type')
            ax.legend(loc='upper right', ncol=2)
            plt.show()
    else:
        print('No event-type column was detected in control_events.csv.')
else:
    print('control_events.csv not available.')


## Section 8 — Game Trace and Game Events

If `game_trace.csv` or `game_events.csv` is available, the notebook summarizes game behavior as the final demonstration layer. Because columns may vary across sessions, the notebook auto-detects likely position, score, and action columns cautiously.


In [ ]:
def find_columns_by_terms(df, include_terms):
    matches = []
    for column in df.columns:
        lower = str(column).lower()
        if any(term in lower for term in include_terms):
            matches.append(column)
    return matches


if 'game_trace.csv' in BUNDLE_OBJECTS:
    game_trace_df = BUNDLE_OBJECTS['game_trace.csv'].copy()
    time_col = find_time_column(game_trace_df)
    position_cols = find_columns_by_terms(game_trace_df, ['x', 'y', 'position', 'cursor', 'player'])
    score_cols = find_columns_by_terms(game_trace_df, ['score', 'points', 'hits', 'miss'])

    summary_rows = []
    if time_col is not None:
        numeric_time = pd.to_numeric(game_trace_df[time_col], errors='coerce')
        summary_rows.append({'Metric': 'Approximate game duration', 'Value': float(numeric_time.max() - numeric_time.min()) if numeric_time.notna().any() else np.nan})
    for column in score_cols[:4]:
        series = pd.to_numeric(game_trace_df[column], errors='coerce')
        if series.notna().any():
            summary_rows.append({'Metric': f'Final {column}', 'Value': float(series.dropna().iloc[-1])})
    if summary_rows:
        display(pd.DataFrame(summary_rows))

    print('Detected game_trace columns:')
    display(pd.DataFrame({'Position-like Columns': [', '.join(position_cols) or 'None'], 'Score-like Columns': [', '.join(score_cols) or 'None']}))

    if time_col is not None and position_cols:
        fig, ax = plt.subplots(figsize=(14, 5))
        for column in position_cols[:3]:
            series = pd.to_numeric(game_trace_df[column], errors='coerce')
            if series.notna().any():
                ax.plot(pd.to_numeric(game_trace_df[time_col], errors='coerce'), series, label=column, linewidth=1.3)
        ax.set_title('Game Trace Over Time')
        ax.set_xlabel(time_col)
        ax.set_ylabel('Detected value')
        ax.legend(loc='upper right')
        plt.show()
else:
    print('game_trace.csv not available.')

if 'game_events.csv' in BUNDLE_OBJECTS:
    game_events_df = BUNDLE_OBJECTS['game_events.csv'].copy()
    time_col = find_time_column(game_events_df)
    event_col = find_event_type_column(game_events_df)
    if event_col is not None:
        event_summary_df = game_events_df.groupby(event_col).size().reset_index(name='Count').rename(columns={event_col: 'Game Event'})
        display(event_summary_df)
    else:
        print('No event-type column was detected in game_events.csv.')
else:
    print('game_events.csv not available.')


## Section 9 — Protocol Truth Comparison

If `protocol_truth.csv` is available, the notebook attempts a cautious alignment between prompted actions and detected control events. The language is intentionally conservative: **approximate alignment**, **detected during prompted interval**, or **outside prompted interval**.


In [ ]:
def find_prompt_action_column(df):
    preferred = ['prompted_action', 'expected_action', 'action', 'prompt', 'label', 'movement', 'command']
    lower_map = {str(column).lower(): column for column in df.columns}
    for candidate in preferred:
        if candidate in lower_map:
            return lower_map[candidate]
    for column in df.columns:
        lower = str(column).lower()
        if any(token in lower for token in ['action', 'prompt', 'label', 'movement', 'command']):
            return column
    return None


def find_interval_columns(df):
    lower_map = {str(column).lower(): column for column in df.columns}
    start_col = None
    end_col = None
    duration_col = None
    for candidate in ['start_sec', 'start_time', 'start', 'onset', 'begin']:
        if candidate in lower_map:
            start_col = lower_map[candidate]
            break
    for candidate in ['end_sec', 'end_time', 'end', 'stop', 'offset']:
        if candidate in lower_map:
            end_col = lower_map[candidate]
            break
    for candidate in ['duration_sec', 'duration', 'length']:
        if candidate in lower_map:
            duration_col = lower_map[candidate]
            break
    return start_col, end_col, duration_col


if 'protocol_truth.csv' in BUNDLE_OBJECTS and 'control_events.csv' in BUNDLE_OBJECTS:
    protocol_df = BUNDLE_OBJECTS['protocol_truth.csv'].copy()
    control_events_df = BUNDLE_OBJECTS['control_events.csv'].copy()

    action_col = find_prompt_action_column(protocol_df)
    start_col, end_col, duration_col = find_interval_columns(protocol_df)
    event_col = find_event_type_column(control_events_df)
    event_time_col = find_time_column(control_events_df)

    if action_col is not None and start_col is not None and event_col is not None and event_time_col is not None:
        truth_df = protocol_df.copy()
        truth_df['__start'] = pd.to_numeric(truth_df[start_col], errors='coerce')
        if end_col is not None:
            truth_df['__end'] = pd.to_numeric(truth_df[end_col], errors='coerce')
        elif duration_col is not None:
            truth_df['__end'] = truth_df['__start'] + pd.to_numeric(truth_df[duration_col], errors='coerce')
        else:
            truth_df['__end'] = truth_df['__start']

        events_df = control_events_df.copy()
        events_df['__time'] = pd.to_numeric(events_df[event_time_col], errors='coerce')

        alignment_rows = []
        for row in truth_df.itertuples(index=False):
            start_time = getattr(row, '__start')
            end_time = getattr(row, '__end')
            action_value = getattr(row, action_col)
            if not np.isfinite(start_time) or not np.isfinite(end_time):
                continue
            detected = events_df.loc[(events_df['__time'] >= start_time) & (events_df['__time'] <= end_time), event_col].astype(str).tolist()
            if detected:
                note = 'detected during prompted interval'
            else:
                nearby = events_df.loc[(events_df['__time'] >= start_time - 0.5) & (events_df['__time'] <= end_time + 0.5), event_col].astype(str).tolist()
                note = 'outside prompted interval' if nearby else 'no aligned detection observed'
            alignment_rows.append(
                {
                    'Prompted Action': action_value,
                    'Expected Window': f"{start_time:.2f} to {end_time:.2f}",
                    'Detected Events': ', '.join(detected[:6]) if detected else 'None',
                    'Notes': note,
                }
            )

        if alignment_rows:
            display(pd.DataFrame(alignment_rows))
        else:
            print('Protocol truth was present, but an approximate alignment table could not be built.')
    else:
        print('Protocol truth comparison skipped because key columns could not be detected reliably.')
else:
    print('protocol_truth.csv and/or control_events.csv not available for comparison.')


## Section 10 — Raw Data Replay Option

If raw OpenBCI recordings are uploaded, the notebook can do a simplified replay-style inspection:

- parse signal channels and marker columns
- estimate sampling rate
- identify likely signal columns
- optionally inspect uploaded model artifacts

However, the notebook will **not** pretend to reproduce the exact runtime unless the necessary models and preprocessing details are present.


In [ ]:
SIGNAL_NAME_RE = re.compile(r'^(ch(?:annel)?|eeg|emg|exg|adc)[ _-]*\d+$', re.IGNORECASE)
INDEX_HINTS = ('sample', 'index')
MARKER_HINTS = ('marker', 'event', 'trigger', 'label', 'stim', 'class')


def looks_like_header(tokens):
    joined = ' '.join(token.strip() for token in tokens)
    return any(char.isalpha() for char in joined)


def guess_delimiter(lines, sample_count=25):
    candidates = [',', '\t', ';']
    scores = {}
    sample = lines[:sample_count]
    for delimiter in candidates:
        scores[delimiter] = sum(line.count(delimiter) for line in sample)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ','


def find_data_start(lines, delimiter):
    for idx, line in enumerate(lines[:80]):
        tokens = [token.strip() for token in line.split(delimiter)]
        if len(tokens) < 4:
            continue
        numeric_like = 0
        for token in tokens:
            try:
                float(token)
                numeric_like += 1
            except Exception:
                pass
        if numeric_like >= max(4, len(tokens) // 2):
            return idx
    return 0


def make_unique_columns(columns):
    counts = Counter()
    out = []
    for idx, name in enumerate(columns):
        candidate = str(name).strip() or f'col_{idx}'
        if candidate.lower().startswith('unnamed'):
            candidate = f'col_{idx}'
        if candidate in counts:
            counts[candidate] += 1
            candidate = f'{candidate}_{counts[candidate]}'
        else:
            counts[candidate] = 0
        out.append(candidate)
    return out


def normalize_column_name(name):
    return re.sub(r'[^a-z0-9]+', '_', str(name).strip().lower()).strip('_')


def detect_raw_columns(df):
    normalized = {column: normalize_column_name(column) for column in df.columns}
    marker_col = None
    signal_cols = [column for column in df.columns if SIGNAL_NAME_RE.match(normalized[column])]
    if not signal_cols:
        numeric_candidates = []
        for column in df.columns:
            numeric = pd.to_numeric(df[column], errors='coerce')
            if numeric.notna().mean() >= 0.80 and numeric.dropna().nunique() >= 25:
                numeric_candidates.append(column)
        signal_cols = numeric_candidates[:8]
    for column in reversed(list(df.columns)):
        lower = normalized[column]
        if any(token in lower for token in MARKER_HINTS):
            marker_col = column
            break
    return signal_cols, marker_col


def parse_raw_recording(file_bytes, filename):
    text = file_bytes.decode('utf-8', errors='ignore').replace('\r\n', '\n').replace('\r', '\n')
    lines = [line for line in text.split('\n') if line.strip()]
    if not lines:
        raise ValueError(f'{filename} is empty.')
    delimiter = guess_delimiter(lines)
    data_start = find_data_start(lines, delimiter)
    tokens = [token.strip() for token in lines[data_start].split(delimiter)]
    header_row = None
    if data_start > 0:
        prev_tokens = [token.strip() for token in lines[data_start - 1].split(delimiter)]
        if len(prev_tokens) == len(tokens) and looks_like_header(prev_tokens):
            header_row = data_start - 1
    if header_row is None and looks_like_header(tokens):
        header_row = data_start
    parse_start = header_row if header_row is not None else data_start
    parse_text = '\n'.join(lines[parse_start:])
    read_kwargs = {'sep': delimiter, 'engine': 'python'}
    if header_row is None:
        read_kwargs['header'] = None
    df = pd.read_csv(io.StringIO(parse_text), **read_kwargs)
    df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
    if header_row is None:
        df.columns = make_unique_columns([f'col_{idx}' for idx in range(df.shape[1])])
    else:
        df.columns = make_unique_columns(df.columns.tolist())
    signal_cols, marker_col = detect_raw_columns(df)
    return {'filename': filename, 'rows': len(df), 'columns': list(df.columns), 'signal_cols': signal_cols, 'marker_col': marker_col}


RAW_RECORDING_FILES = FILE_GROUPS.get('raw recording', [])
if RAW_RECORDING_FILES:
    raw_summaries = []
    for filename in RAW_RECORDING_FILES:
        try:
            raw_summaries.append(parse_raw_recording(UPLOADED_FILES[filename], filename))
        except Exception as exc:
            raw_summaries.append({'filename': filename, 'rows': np.nan, 'columns': [], 'signal_cols': [], 'marker_col': f'Parse failed: {exc}'})
    raw_summary_df = pd.DataFrame([
        {
            'Filename': row['filename'],
            'Rows': row['rows'],
            'Signal Columns': ', '.join(row['signal_cols']) if row['signal_cols'] else 'None detected',
            'Marker Column': row['marker_col'] if isinstance(row['marker_col'], str) else (row['marker_col'] or 'None detected'),
        }
        for row in raw_summaries
    ])
    display(raw_summary_df)
else:
    print('No raw recording files were identified for replay inspection.')

MODEL_ARTIFACT_LOADS = []
for artifact_file in FILE_GROUPS.get('model artifact', []):
    try:
        obj = pickle.loads(UPLOADED_FILES[artifact_file])
        MODEL_ARTIFACT_LOADS.append({'Filename': artifact_file, 'Loaded Type': type(obj).__name__})
    except Exception as exc:
        MODEL_ARTIFACT_LOADS.append({'Filename': artifact_file, 'Loaded Type': f'Could not load ({exc})'})

if MODEL_ARTIFACT_LOADS:
    print('Uploaded model artifact inspection:')
    display(pd.DataFrame(MODEL_ARTIFACT_LOADS))

replay_note = """
### Raw Replay Note

This notebook can inspect uploaded raw recordings and uploaded model files, but it does not claim to reproduce the exact live runtime unless the full preprocessing contract, feature contract, and saved model artifacts are available and aligned. For exact runtime behavior, the live-run logs remain the most trustworthy evidence.
"""
display(Markdown(replay_note))


## Section 11 — Final System Interpretation

The interpretation below keeps the project-specific claims grounded in what the branch actually does.


In [ ]:
interpretation = """
### Final System Interpretation

**What does the final system actually do?**  
It combines a stronger jaw-based click branch with a weaker EEG left/right directional branch inside one runtime. Continuous predictions are stabilized through calibration, adaptation, thresholds, smoothing, and cooldown before they become game actions.

**Which parts are strongest?**  
The jaw model is the strongest reusable artifact in the system. It is the most practical source of discrete click-style control.

**Which parts are still experimental?**  
The EEG left/right model remains the weaker branch and should be reported honestly. It contributes directional intent research value, but it is more sensitive to session drift and calibration quality.

**Why is the hybrid design more realistic than EEG-only control?**  
Because it lets the system rely on the strongest available signal for discrete control while keeping EEG in a more realistic directional role instead of forcing it to do everything.

**What did calibration and adaptation add?**  
They made the live runtime more stable by aligning thresholds, margins, and decision logic to the current session instead of assuming one universal fixed operating point.

**What would the next retraining step be?**  
The next logical retraining step would be to use structured LRJ/guided event sessions as future training data, especially for improving event-based left/right decoding. The current final branch should be described as **fixed saved models plus session-specific calibration/adaptation**, not full live retraining.

**How does this connect to the GUI/game?**  
The GUI/game is the final demonstration layer that turns the hybrid control logic into a usable interaction system.
"""
display(Markdown(interpretation))


## Section 12 — Final Takeaway

This project began as an EEG-only interaction classifier but evolved into a hybrid BCI system after benchmarking showed that jaw activity provided a stronger discrete control signal than left/right EEG direction decoding. The final system uses jaw detection for reliable click actions, preserves EEG left/right as a directional intent branch, and stabilizes live behavior with calibration, thresholds, smoothing, and cooldown logic.
